# Correcting a ring's orbit through the virtual accelerator

A tool built in a notebook, used on the virtual accelerator as it would be on the machine (ADR 0008,
ADR 0010): read the BPMs, compute a correction with the model, write the correctors as one logged
change, and check what the BPMs read afterwards. It works on a sandbox, your own copy of a shared
ring, so nobody else's machine changes.

**Before you start.** A virtual accelerator must be running, and the notebook needs a token of yours:

```
eicvibe-web EIN/machine --va STATE           # the server, with the virtual accelerator
eicvibe-va token STATE yourname              # a token for this notebook (or the admin page's API tokens)
export EICVIBE_URL=http://localhost:8000 EICVIBE_TOKEN=...
```

Its ring needs errors to correct (the admin page's Errors: misaligned quadrupoles, say).
`EICVIBE_MACHINE` is the machine folder the server runs, for the model.

In [1]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

import eicvibe.notebook as ev
from eicvibe.engines import Effect
from eicvibe.operations.model import MachineModel
from eicvibe.operations.orbit import correct_orbit, orbit_response
from eicvibe.va.settings import MachineSettings

MACHINE = Path(os.environ.get("EICVIBE_MACHINE", "../../virtual_accelerator/EIN/machine"))
va = ev.Client()  # EICVIBE_URL and EICVIBE_TOKEN
me = va.me()
print(f"{me['name']} ({me['role']}) on {va.url}")
pd.DataFrame(va.machines()).set_index("id")[["name", "kind", "target", "state", "shared", "owner"]]

URLError: <urlopen error [Errno 61] Connection refused>

## A sandbox of the ring

The shared ring to start from: `EICVIBE_VA_MACHINE`, or the first shared ring running. Its sandbox has
its settings as they are now, and its errors.

In [ ]:
machines = va.machines()
shared = os.environ.get("EICVIBE_VA_MACHINE") or next(
    x["id"] for x in machines if x["kind"] == "ring" and x["shared"] and x["state"] == "running"
)
box = va.fork(shared, name=f"orbit-{me['name']}-{time.strftime('%H%M%S')}", reason="orbit correction study")
va.start(box["id"], reason="to correct its orbit")
while (box := va.machine(box["id"]))["state"] != "running" or (box["position"] or 0) < 1024:
    time.sleep(1)  # started, and some turns tracked
print(f"sandbox {box['id']} of {shared}: {box['target']}, running")

## What the BPMs read

The orbit is the mean of the readings over a few hundred turns: the betatron oscillation left from
injection averages out. The readings are what a BPM system gives: each BPM's hidden offset and gain
included.

In [ ]:
def measure(machine: str, frames: int = 40, every: int = 10) -> pd.DataFrame:
    got = va.orbits(machine, every=every, count=frames)
    x, y = (np.nanmean(np.array(got[p], dtype=float), axis=0) for p in ("x", "y"))
    return pd.DataFrame({"s": got["s"], "x": x, "y": y}, index=pd.Index(got["bpms"], name="bpm"))


before = measure(box["id"])
print(f"rms x {before.x.std():.3f} mm, y {before.y.std():.3f} mm at {len(before)} BPMs")
ev.chart({"x": (before.s, before.x), "y": (before.s, before.y)}, title="Orbit before", x_label="s (m)",
         y_label="mm", style="both", names=list(before.index))

## The model

The model is the machine as it is set: the machine files' design (the same folder the server runs),
the sandbox's operating mode, and its live settings, read through the client. It knows nothing of the
hidden errors: those are what the BPMs see. Its response matrix (BPMs × correctors) comes from the
closed orbit with each corrector stepped in turn: a minute for the RCS's 196 correctors a plane.

In [ ]:
m = ev.machine(MACHINE)
settings = MachineSettings(m.lattice, m.registry, ev.mode_values(m, box["mode"]) if box.get("mode") else {})
live = {channel: row["value"] for channel, row in va.settings(box["id"], changed=True).items()}
model = MachineModel(settings, box["target"], live, options=ev.Options(effects={Effect.LONGITUDINAL}))
response = orbit_response(model, "x")
print(f"{len(model.bpms)} BPMs, {len(model.correctors['x'])} horizontal and {len(model.correctors['y'])} vertical correctors")
singular = np.linalg.svd(response.matrix, compute_uv=False)
ev.chart({"singular values": (np.arange(len(singular)), singular)}, title="The horizontal response's singular values",
         x_label="index", y_label="m per A", style="points", log=(np.floor(np.log10(singular.min())), 0))

## A correction, written here

Least squares on the response matrix, with the small singular values damped (Tikhonov): the change
of every corrector, in its channel's units.

In [ ]:
def damped_svd(matrix: np.ndarray, orbit: np.ndarray, damping: float = 0.01) -> np.ndarray:
    u, s, vt = np.linalg.svd(matrix, full_matrices=False)
    lam = damping * s[0]
    return -(vt.T * (s / (s**2 + lam**2))) @ (u.T @ orbit)


orbit_x = before.loc[list(response.bpms), "x"].to_numpy() * 1e-3  # m, in the response's BPM order
mine = damped_svd(response.matrix, orbit_x)
predicted = orbit_x + response.matrix @ mine
print(f"predicted rms x: {orbit_x.std() * 1e3:.3f} -> {predicted.std() * 1e3:.3f} mm, largest change {abs(mine).max():.3g} A")

## The vetted operation

`correct_orbit` is the operation the viewer's Operate tab and the AI tools use (ADR 0010): the same
method, with the damping raised until every corrector is within its limits, and checks of its
prediction made in the model. It proposes; nothing is written yet.

In [ ]:
orbit = {plane: dict(zip(before.index, before[plane] * 1e-3)) for plane in ("x", "y")}
proposals = {plane: correct_orbit(model, plane, orbit[plane]) for plane in ("x", "y")}
for proposal in proposals.values():
    print(proposal.summary)
    for check in proposal.checks:
        print(f"  {'ok' if check.ok else 'FAILS'}  {check.name}: {check.note}")

In [ ]:
theirs = np.array([c.change for c in proposals["x"].changes])
ev.chart({"written here": (np.arange(len(mine)), mine), "correct_orbit": (np.arange(len(theirs)), theirs)},
         title="Horizontal corrector changes, both ways", x_label="corrector (beam order)", y_label="A", style="points")

## Written, as one change, and checked

The sandbox's settings change as one logged change (with its reason); the correctors ramp to their
new values over the machine's `ramp_turns`, as magnets follow their supplies. Then the BPMs again.

In [ ]:
change = proposals["x"].values() | proposals["y"].values()
written = va.set(box["id"], change, reason="orbit correction from a notebook")
log = va.log(correlation=written["correlation"])
turn = log[0]["turn"]  # the run's turn when it was written
while va.machine(box["id"])["position"] < turn + box["ramp_turns"] + 1500:  # the ramp, then turns to measure
    time.sleep(1)
after = measure(box["id"])
pd.DataFrame({
    "before (mm)": before[["x", "y"]].std(),
    "predicted (mm)": [proposals[p].predicted["rms"][1] for p in ("x", "y")],
    "after (mm)": after[["x", "y"]].std(),
})

The BPMs agree with the prediction to a few tens of microns: the model knows nothing of the hidden
errors, and the BPMs read with noise and their own offsets. Measuring and correcting again takes the
orbit closer, as in a control room.

In [ ]:
ev.chart({"x before": (before.s, before.x), "x after": (after.s, after.x)}, title="Horizontal orbit",
         x_label="s (m)", y_label="mm", style="both", names=list(after.index))

In [ ]:
ev.chart({"y before": (before.s, before.y), "y after": (after.s, after.y)}, title="Vertical orbit",
         x_label="s (m)", y_label="mm", style="both", names=list(after.index))

The change in the log, as anyone with the admin page sees it:

In [ ]:
pd.DataFrame(log)[["time", "user", "machine", "channel", "old", "new", "unit", "turn", "reason"]].head()

Done: the sandbox goes (its run's folder stays with the server, for the viewer's Orbit tab).

In [ ]:
model.close()
va.stop(box["id"], reason="done")
while va.machine(box["id"])["state"] not in ("stopped", "failed"):
    time.sleep(1)
va.remove(box["id"], reason="orbit correction study done")

## Next

- A bump, or a tune move: `eicvibe.operations.bump.local_bump`, `eicvibe.operations.tune.move_tunes`.
- MICADO, for a single error: `correct_orbit(model, "x", orbit, method="micado", count=3)`.
- On a shared ring, an operator applies; a physicist proposes through the viewer's Operate tab.